# Clase 08 — Una red pequeña en PyTorch

Procesamiento de Lenguaje Natural - Felipe Castillo Rendón

Versión alumno. Las demostraciones explican el mecanismo; el reto pide una interpretación propia. Consulta el enunciado del laboratorio evaluable para conservar alcance y tiempo.

## Resumen general de la clase (comentario del profesor)

**Idea central:** construir y entender una **red neuronal mínima en PyTorch** que clasifica solicitudes de soporte en 4 clases (`acceso`, `conectividad`, `pagos`, `software`): embeddings → media con máscara → capa lineal → logits.

**Recorrido del notebook**
1. **Tensores y formas** (B textos, L=32 posiciones, d=24 coordenadas, C=4 clases): `4×32 → 4×32×24 → 4×24 → 4×4`. Los IDs de palabra son índices, no «importancia».
2. **Padding + máscara:** se rellena a 32 posiciones; la media debe dividir entre las palabras **reales**, no entre 32 (en el ejemplo, [1,2] y no [0.67, 1.33]).
3. **Logits → softmax → entropía cruzada:** `cross_entropy` recibe logits (nunca softmax previo); con p=0.75 la pérdida es −ln(0.75)=0.288.
4. **Gradiente y descenso:** (w−3)² con w=1 → gradiente −4 → w=1.4 → pérdida de 4 a 2.56. El ciclo de entrenamiento es `zero_grad → forward → loss → backward → step`.
5. **Train / validation / test:** train ajusta pesos, validation elige la época (*early stopping*), test queda reservado. Se restauró la época 7 (val. 1.303) aunque train siguió bajando hasta 0.444: **sobreajuste**.
6. **Resultado honesto:** 8/24 aciertos (33 %) y F1 macro 0.24 en validation. El modelo casi todo lo manda a «software». Hipótesis (por verificar): en validation aparecen palabras que no están en train → `UNK`, y la partición es por *familias* de mensajes, así que no hay mensajes casi idénticos que memorizar.
7. **Sobreajustar 4 ejemplos** (100 %) solo prueba que el circuito de entrenamiento funciona; **no** mide generalización.

**E1 (completado abajo):** comparar contra una referencia clásica (TF-IDF + regresión logística) en la misma validación y quitar la máscara en una copia para ver cuánto cambia el promedio de un texto corto.

> Nota: las celdas que yo agregué no tienen salida guardada. Sube `solicitudes.csv` y `apoyo_pln.py` en Colab y ejecuta todo para ver las cifras.


## Archivos que acompañan al notebook

Vamos a trabajar con `solicitudes.csv`, que contiene los mensajes, y `apoyo_pln.py`, que contiene las funciones de apoyo. Ambos archivos van al mismo nivel que el notebook; no se necesitan carpetas ni un ZIP de recursos.

En Colab, abre el notebook y ejecuta la siguiente celda. Selecciona ambos archivos cuando aparezca el botón de carga. También puedes subirlos desde el panel Archivos, directamente a la carpeta inicial de la sesión. Si ya están cargados, la celda los reutiliza.



In [1]:
#@title Cargar los archivos de la práctica
from pathlib import Path
import sys

# El CSV y las funciones se buscan en la carpeta actual de la sesión.
ROOT = Path.cwd()
necesarios = ("solicitudes.csv", "apoyo_pln.py")
faltantes = [nombre for nombre in necesarios if not (ROOT / nombre).is_file()]
if faltantes:
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError("Coloca junto al notebook: " + ", ".join(faltantes)) from None
    print("Selecciona los archivos que faltan: " + ", ".join(faltantes))
    files.upload()
faltantes = [nombre for nombre in necesarios if not (ROOT / nombre).is_file()]
if faltantes:
    raise FileNotFoundError("Todavía falta cargar: " + ", ".join(faltantes))
# La ruta explícita permite importar el apoyo sin instalar un paquete del curso.
sys.path.insert(0, str(ROOT))
print("Listo: solicitudes.csv y apoyo_pln.py están en la carpeta de trabajo.")


Listo: solicitudes.csv y apoyo_pln.py están en la carpeta de trabajo.


## Entorno de ejecución

Usaremos NumPy, pandas, scikit-learn, Matplotlib y PyTorch. En una instalación local con Python 3.11 puedes instalar las dependencias de `requirements_clases_07_10.txt` y seleccionar ese entorno como kernel. En Colab se utilizan las bibliotecas de la sesión; la siguiente celda muestra sus versiones.

La práctica utiliza CPU y no necesita una clave de acceso ni descargar un modelo. El modo `minimo` conserva el recorrido de clase. Una semilla fija el inicio del azar, aunque las versiones pueden producir pequeñas diferencias numéricas.


In [2]:
from pathlib import Path
import sys, os, platform, time
from importlib.metadata import version
import numpy as np
import pandas as pd
import torch
from IPython.display import display
# Los recursos se cargaron en la misma carpeta que este notebook.
from apoyo_pln import load_requests, seed_all, words, LABELS
seed_all()
MODE=os.environ.get('PLN_MODE','minimo')
assert MODE in {'minimo','completo'}
pd.set_option('display.max_colwidth',150)
print('Python:',platform.python_version(),'Modo:',MODE,'Dispositivo: CPU')
print({p:version(p) for p in ['numpy','pandas','scikit-learn','torch','matplotlib']})

Python: 3.12.10 Modo: minimo Dispositivo: CPU
{'numpy': '2.5.3', 'pandas': '3.0.5', 'scikit-learn': '1.9.0', 'torch': '2.15.0', 'matplotlib': '3.11.1'}


La salida identifica el entorno que está ejecutando el código. Todavía no hemos cargado el corpus ni entrenado una red. Si falta un recurso, prepara la carpeta antes de continuar. Guarda los resultados antes de cerrar una sesión temporal.

### Tensor: números con una forma

**De listas a dimensiones.** Un tensor es un arreglo de números. Un vector tiene un eje; una matriz, dos. PyTorch guarda la forma en .shape para comprobar qué representa cada eje.

**Nuestras letras.** B: textos del lote; L: posiciones por texto; V: palabras del vocabulario; d: coordenadas del embedding; C: clases. Un lote agrupa ejemplos procesados juntos.

Los IDs son índices para buscar filas; no indican cercanía de significado.

### Sigamos el ejemplo que sí ejecutaremos
B es la cantidad de textos del lote: 4. L es la longitud reservada: 32 posiciones. d es la cantidad de coordenadas por palabra: 24. C es la cantidad de clases: 4.

Los índices tienen forma 4×32. La búsqueda de embeddings produce 4×32×24: texto, posición, coordenada. La media con máscara deja 4×24 y la capa final devuelve 4×4: texto, categoría. V nombra el tamaño del vocabulario, que comprobaremos al cargar train.

Un índice de palabra sirve para buscar una fila; no mide cercanía ni importancia.

### Padding: rellenar no añade palabras

**Por qué se utiliza.** Los textos tienen longitudes distintas; un lote necesita una forma común. Padding añade posiciones de relleno, identificadas aquí por ID 0.

**Máscara.** Una máscara indica qué posiciones son reales. Con IDs [4,7,0], la máscara es [1,1,0]. El 0 evita que el relleno participe como palabra.

Esta máscara de padding difiere de la máscara causal de la clase 10.

### Pooling: reunir varias posiciones

**Valores elegidos.** Dos palabras tienen vectores [2,0] y [0,4]; el padding tiene [0,0]. Pooling reúne información de varias posiciones en una representación.

**Media con máscara.** Suma: [2,0]+[0,4]+[0,0]=[2,4].
Hay 2 posiciones válidas.
[2,4]/2=[1,2].
Dividir entre 3 daría [0.6667,1.3333].

El denominador cuenta palabras válidas, no longitud rellenada.

## 1. Media con máscara

El último vector es relleno. Multiplicamos por la máscara, sumamos posiciones y dividimos entre las válidas.

**Aquí recibimos:** e: tensor 1×3×2; un texto, tres posiciones y dos coordenadas. mask contiene 1, 1, 0.

**Lo que haremos:** unsqueeze añade un eje para aplicar la máscara a ambas coordenadas. sum(1) reúne posiciones; dividimos por las válidas.

**Antes de ejecutar:** ¿El relleno debe contar como una tercera palabra?

In [3]:
e=torch.tensor([[[2.,0.],[0.,4.],[0.,0.]]])
mask=torch.tensor([[1,1,0]]).unsqueeze(-1)
pooled=(e*mask).sum(1)/mask.sum(1)
print("Media correcta:",pooled,"Sin máscara:",e.mean(1))
assert torch.allclose(pooled,torch.tensor([[1.,2.]]))

Media correcta: tensor([[1., 2.]]) Sin máscara: tensor([[0.6667, 1.3333]])


### Qué nos dice esta ejecución

La suma por coordenada es [2+0+0,0+4+0]=[2,4]. La máscara suma 1+1+0=2. Dividimos y obtenemos [1,2]. Sin máscara, [2/3,4/3]≈[0.6667,1.3333]. El texto no cambió, solo se contó indebidamente una posición vacía.

**Interpretación.** Sin máscara se divide entre tres aunque solo haya dos palabras: el relleno reduce indebidamente la magnitud.

### Clasificador lineal: una cuenta visible

**Pesos y entrada.** h=[1,2], W=[[1,0],[0,1]] y b=[0.5,−0.5]. W contiene pesos aprendibles; b añade un ajuste llamado sesgo.

**Salida.** z=hWᵀ+b.
z₁=1×1+2×0+0.5=1.5.
z₂=1×0+2×1−0.5=1.5.
Estos logits no son probabilidades.

nn.Linear aplica esta transformación; aquí usamos dos salidas para facilitar la cuenta.

### Softmax: transformar los logits

**Las operaciones.** exp(zᵢ)=e elevado a zᵢ. Sumamos esas exponenciales y dividimos cada una entre el total. El índice i selecciona una clase.

**Un segundo ejemplo, elegido para calcular a mano.** z=[ln(3),0]≈[1.098612,0].
exp(z)=[3,1]; suma=4.
p=[3/4,1/4]=[0.75,0.25].
Python: torch.softmax(z, dim=−1).

−1 selecciona el último eje: aquí, las clases.

### Entropía cruzada: medir el error

**Qué entra en la pérdida.** Para una etiqueta única usamos −ln(p de la clase correcta). ln(x) pregunta a qué potencia elevar e para obtener x; e≈2.71828.

**Completar el ejemplo.** Si la clase correcta es 0: p=0.75.
ln(0.75)≈−0.287682.
Pérdida=0.287682.
Si fuera 1: −ln(0.25)≈1.386294.

cross_entropy recibe logits y etiquetas enteras; no apliques softmax antes.

## 2. Logits, softmax y pérdida

Números elegidos para enseñar. Primero cada operación; después comprobamos la combinación implementada por cross_entropy.

**Aquí recibimos:** h, W y b, números didácticos para una capa lineal; después z, otro ejemplo de logits para softmax.

**Lo que haremos:** Primero mostramos una transformación. Luego cambiamos expresamente de ejemplo para estudiar probabilidades y pérdida.

**Antes de ejecutar:** Si aumenta la probabilidad de la clase correcta, ¿qué debería pasar con la pérdida?

In [4]:
h=torch.tensor([[1.,2.]]);W=torch.eye(2);b=torch.tensor([.5,-.5])
print("Transformación lineal:",h@W.T+b)
z=torch.tensor([[3.,1.]]).log();target=torch.tensor([0])
exp_z=z.exp();prob=exp_z/exp_z.sum(-1,keepdim=True)
manual=-prob[0,0].log()
library=torch.nn.functional.cross_entropy(z,target)
print("Logits:",z,"Exp:",exp_z,"Probabilidades:",prob,"Pérdida:",manual.item())
assert torch.allclose(manual,library)

Transformación lineal: tensor([[1.5000, 1.5000]])
Logits: tensor([[1.0986, 0.0000]]) Exp: tensor([[3., 1.]]) Probabilidades: tensor([[0.7500, 0.2500]]) Pérdida: 0.28768205642700195


### Qué nos dice esta ejecución

Primer ejemplo: z₀=1×1+2×0+0.5=1.5 y z₁=1×0+2×1−0.5=1.5. El segundo ejemplo usa otros logits elegidos: [ln(3),ln(1)]≈[1.098612,0]. ln es logaritmo natural, base e≈2.71828: ln(3) pregunta qué exponente de e produce 3; np.log(3) lo calcula. Al aplicar exp recuperamos [3,1], sumamos 4 y dividimos: [0.75,0.25].

Para etiqueta 0, pérdida=−ln(0.75)=−(−0.287682)=0.287682. Para etiqueta 1 sería −ln(0.25)=1.386294. La función cross_entropy recibe los logits originales; aplicar softmax antes alteraría su entrada. Los valores que suman uno tampoco prueban que el modelo esté calibrado.

**Interpretación.** Con clase correcta 0: −ln(0.75)≈0.287682. La biblioteca recibe z, no prob.

### Gradiente: sensibilidad local

**Una pérdida sencilla.** L(w)=(w−3)². w es un peso; L mide su distancia cuadrática a 3. La derivada dL/dw=2(w−3) indica cómo cambia L al mover ligeramente w.

**Evaluar en w=1.** L(1)=(1−3)²=4.
Gradiente=2(1−3)=−4.
El signo negativo indica que un pequeño aumento de w reduce esta pérdida.

Autograd calcula derivadas enlazando las operaciones del código.

### Actualizar el peso: sustituir y calcular

**La regla.** w_nuevo=w−η·gradiente.
η, eta, es la tasa de aprendizaje: controla el tamaño del paso. Elegimos η=0.1.

**La comprobación.** w_nuevo=1−0.1×(−4)=1.4.
L(1.4)=(−1.6)²=2.56.
Bajamos de 4 a 2.56. Un paso excesivo podría aumentar la pérdida.

La cuenta enseña descenso simple; Adam ajusta los pasos utilizando su historial.

### El orden del ciclo de entrenamiento

**Antes de actualizar.** zero_grad borra gradientes acumulados.
forward calcula predicciones.
La pérdida compara predicciones y etiquetas.

**Calcular y mover.** backward calcula gradientes.
step usa esos gradientes para actualizar los pesos.
train() por sí solo no ejecuta ninguna de estas actualizaciones.

No reutilices por accidente gradientes de un paso anterior.

## 3. Comprobar un gradiente

requires_grad registra derivadas. backward calcula sensibilidad; no_grad evita registrar la actualización.

**Aquí recibimos:** w=1, un peso escalar; pérdida L(w)=(w−3)².

**Lo que haremos:** backward calcula el gradiente. Aplicamos un paso de tamaño 0.1 fuera del registro de derivadas.

**Antes de ejecutar:** ¿Qué indica el signo negativo del gradiente en este punto?

In [4]:
w=torch.tensor(1.,requires_grad=True)
loss=(w-3)**2;loss.backward()
gradient=w.grad.item();before=loss.item()
with torch.no_grad():w-=.1*w.grad
print(dict(gradiente=gradient,peso_nuevo=w.item(),antes=before,despues=((w-3)**2).item()))
assert abs(gradient+4)<1e-6 and abs(w.item()-1.4)<1e-6

{'gradiente': -4.0, 'peso_nuevo': 1.399999976158142, 'antes': 4.0, 'despues': 2.56000018119812}


### Qué nos dice esta ejecución

La derivada de (w−3)² es 2(w−3). En w=1 vale −4; este signo indica que aumentar un poco w reduce la pérdida. Actualizamos 1−0.1×(−4)=1.4. Antes: (1−3)²=4. Después: (1.4−3)²=2.56. La reducción es 1.44. Es una cuenta de descenso simple; Adam ajusta pasos usando historial de gradientes y no se reduce a esta misma regla.

**Interpretación.** Gradiente −4; peso de 1 a 1.4; pérdida de 4 a 2.56. Es descenso simple; después usamos Adam.

## 4. Ver la red por dentro

Reconstruimos forward: embeddings, máscara, media y proyección. La función del curso debe coincidir.

**Aquí recibimos:** Cuatro textos de train, un vocabulario de palabras y una red todavía sin entrenar.

**Lo que haremos:** Encode convierte palabras en índices y rellena hasta 32 posiciones. Mostramos cada transformación de forward, el recorrido hacia la salida.

**Antes de ejecutar:** ¿Las cuatro columnas finales representan palabras o categorías?

In [5]:
from apoyo_pln import vocabulary
# ID 0 reserva relleno; ID 1 representa palabras ausentes del vocabulario.
def encode(texts,vocab,max_length=32):
    encoded=torch.zeros((len(texts),max_length),dtype=torch.long)
    for row,text in enumerate(texts):
        ids=[vocab.get(word,1) for word in words(text)[:max_length]]
        encoded[row,:len(ids)]=torch.tensor(ids,dtype=torch.long)
    return encoded

class MeanClassifier(torch.nn.Module):
    def __init__(self,vocab_size,dim=24,n_labels=4):
        super().__init__()
        self.embedding=torch.nn.Embedding(vocab_size,dim,padding_idx=0)
        self.output=torch.nn.Linear(dim,n_labels)
    def forward(self,ids):
        valid=(ids!=0).unsqueeze(-1)
        # Contamos palabras, no las 32 posiciones reservadas en memoria.
        pooled=(self.embedding(ids)*valid).sum(1)/valid.sum(1).clamp(min=1)
        return self.output(pooled)

from apoyo_pln import LABELS
df=load_requests();train=df[df.split=='train'];validation=df[df.split=='validation']
vocab=vocabulary(train.text);ids=encode(list(train.text[:4]),vocab)
network=MeanClassifier(len(vocab))
emb=network.embedding(ids);valid=(ids!=0).unsqueeze(-1)
mean=(emb*valid).sum(1)/valid.sum(1).clamp(min=1)
logits=network.output(mean)
print("IDs:",ids.shape,"Embedding:",emb.shape,"Media:",mean.shape,"Logits:",logits.shape)
assert torch.allclose(network(ids),logits)

print("Vocabulario con PAD y UNK:",len(vocab))
display(train.head(4)[["id","text","label"]])
print("Índices del primer texto:",ids[0].tolist())
print("Longitud máxima observada:",max(len(words(t)) for t in train.text))

IDs: torch.Size([4, 32]) Embedding: torch.Size([4, 32, 24]) Media: torch.Size([4, 24]) Logits: torch.Size([4, 4])
Vocabulario con PAD y UNK: 151


,id,text,label
0,S001,Olvidé la contraseña y necesito restablecerla.,acceso
1,S002,"Buen día, olvidé la contraseña y necesito restablecerla. ¿Me pueden orientar?",acceso
2,S003,"Olvidé la contraseña y necesito restablecerla; ya reinicié y sigue igual, urge tantito.",acceso
3,S004,El código de verificación no llega a mi correo.,acceso


Índices del primer texto: [100, 83, 36, 148, 97, 124, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Longitud máxima observada: 20


### Qué nos dice esta ejecución

Con V=151, las formas reales son 4×32 → 4×32×24 → 4×24 → 4×4. El primer 4 cuenta textos; el último, categorías en el orden LABELS. El ID de solicitud S001 identifica una fila del CSV; los índices dentro del tensor identifican palabras del vocabulario, no solicitudes.

PAD=0 se excluye de la media; UNK=1 se cuenta como posición, aunque no conozcamos esa palabra. encode conserva hasta 32 tokens. Una entrada vacía devolvería una media cero gracias al denominador mínimo 1 y luego el sesgo de la capa: no se debe interpretar como comprensión de un mensaje. Esta red inicia embeddings nuevos de 24 coordenadas; no reutiliza los de 16 de Clase 7.

**Interpretación.** La salida es 4×4: cuatro textos y cuatro clases. clamp(min=1) evita dividir entre cero; no convierte una entrada vacía en lenguaje.

### Validación: elegir sin mirar test

**Separar decisiones.** Train ajusta pesos. Validation ayuda a elegir la época. Test queda reservado para la evaluación final. No elegimos una época revisando test.

**Parada temprana.** Early stopping detiene cuando validación deja de mejorar. Un checkpoint es una copia de pesos; restauramos el mejor, no necesariamente el último.

El helper espera hasta cinco épocas sin mejora de validation_loss.

### Regularización: mecanismos diferentes

**Dos ejemplos.** Dropout desactiva aleatoriamente activaciones durante entrenamiento. Weight decay reduce pesos según la regla del optimizador. No son el mismo mecanismo.

**Qué usa esta práctica.** La red mínima no tiene dropout ni weight decay. Sí selecciona pesos con validación. No atribuyas resultados a un mecanismo ausente del código.

Estas alternativas se explican como consulta; no añaden una entrega.

### eval() y no_grad() no son equivalentes

**Modo de las capas.** train() y eval() cambian capas como dropout. En evaluación, dropout deja de apagar activaciones aleatoriamente.

**Registro de operaciones.** no_grad() evita construir el registro necesario para derivar. Reduce trabajo en inferencia, pero no cambia por sí solo el modo de las capas.

Para predecir usamos model.eval() y un bloque torch.no_grad().

## 5. Preparar train y validation

Comparten el vocabulario de train y el orden de etiquetas LABELS.

**Aquí recibimos:** train con 96 textos y validation con 24; LABELS fija el orden acceso, conectividad, pagos y software.

**Lo que haremos:** x y vx son índices; y y vy son categorías enteras. Creamos la red desde cero con semilla 42 y Adam con tasa 0.035.

**Antes de ejecutar:** ¿Qué conjunto puede elegir la época y cuál permanece reservado?

In [6]:
seed_all()
x=encode(list(train.text),vocab);y=torch.tensor([LABELS.index(z) for z in train.label])
vx=encode(list(validation.text),vocab);vy=torch.tensor([LABELS.index(z) for z in validation.label])
model=MeanClassifier(len(vocab));opt=torch.optim.Adam(model.parameters(),lr=.035)
best=float('inf');best_state=None;patience=0;history=[]
epochs=25 if MODE=='minimo' else 80
start=time.perf_counter()

### Qué nos dice esta ejecución

x organiza 96×32 posiciones y vx, 24×32. y y vy tienen una categoría por texto. La red contiene 151×24+24×4+4=3,624+96+4=3,724 parámetros registrados, incluida la fila de padding. El optimizador recibe train; validation seleccionará pesos. No hay dropout ni penalización de pesos configurados. No atribuyamos resultados a mecanismos que no están aquí.

**Interpretación.** Test no participa. best_state guardará los pesos elegidos mediante validation.

## 6. Entrenar y elegir pesos

Cada época actualiza con train y mide validation sin gradientes. Copiamos pesos cuando mejora; detenemos tras cinco épocas sin mejora.

**Aquí recibimos:** model, x, y, vx y vy; como máximo 25 épocas en modo minimo.

**Lo que haremos:** Actualizamos con train; medimos validation sin gradientes. Guardamos una copia independiente de los mejores pesos y detenemos tras cinco épocas sin mejora suficiente.

**Antes de ejecutar:** ¿La última época será necesariamente la que conservaremos?

In [7]:
for epoch in range(epochs):
    model.train();opt.zero_grad()
    loss=torch.nn.functional.cross_entropy(model(x),y)
    loss.backward();opt.step();model.eval()
    with torch.no_grad():val_loss=torch.nn.functional.cross_entropy(model(vx),vy).item()
    history.append(dict(epoch=epoch+1,train_loss=loss.item(),validation_loss=val_loss))
    if val_loss<best-1e-4:
        best=val_loss;best_state={k:v.detach().clone() for k,v in model.state_dict().items()};patience=0
    else:patience+=1
    if patience>=5:break
model.load_state_dict(best_state);model.eval()
print("Segundos:",time.perf_counter()-start,"Parámetros:",sum(p.numel() for p in model.parameters()))
display(pd.DataFrame(history))

best_epoch=next(row["epoch"] for row in history if np.isclose(row["validation_loss"],best,rtol=0,atol=1e-8))
print("Época restaurada:",best_epoch,"Pérdida restaurada:",best)

Segundos: 3.6613272500107996 Parámetros: 3724


,epoch,train_loss,validation_loss
0,1,1.372267,1.357696
1,2,1.298123,1.337637
2,3,1.227499,1.324353
3,4,1.154946,1.314803
4,5,1.076620,1.307467
5,6,0.991347,1.303182
6,7,0.900129,1.302946
7,8,0.805363,1.306624
8,9,0.709943,1.313069
9,10,0.616475,1.321150


Época restaurada: 7 Pérdida restaurada: 1.302945613861084


### Qué nos dice esta ejecución

La ejecución llegó a la época 12 y restauró la 7. En la 7, validation_loss≈1.302945; en la 12 subió a 1.341576. Tras cinco épocas sin mejora de al menos 0.0001, se detuvo. La pérdida de train siguió bajando: 1.372267 en época 1 y 0.444037 en época 12. Este contraste muestra por qué guardar los últimos pesos sería una mala regla aquí.

La columna epoch cuenta recorridos completos. train_loss se mide antes del paso de actualización y validation_loss después: corresponden al mismo ciclo, pero no exactamente al mismo estado de pesos. Las filas no son porcentajes de errores. Solo validation decide el checkpoint, la copia de pesos a restaurar.

**Interpretación.** La última fila no tiene por qué ser la elegida. Recargamos el mejor checkpoint sin entrenar de nuevo.

## 7. Medir validación

argmax elige el mayor logit; convertimos índices a etiquetas.

**Aquí recibimos:** El modelo restaurado y los 24 mensajes de validation.

**Lo que haremos:** argmax elige la mayor puntuación. La tabla compara etiqueta y predicción; la matriz cuenta sus coincidencias y confusiones.

**Antes de ejecutar:** ¿Podemos llamar a esta cifra una evaluación final independiente?

In [8]:
from sklearn.metrics import f1_score
with torch.no_grad():
    prediction=[LABELS[int(i)] for i in model(vx).argmax(-1)]
    restored_loss=torch.nn.functional.cross_entropy(model(vx),vy).item()
print("F1 macro validation:",f1_score(validation.label,prediction,average='macro'))
assert np.isclose(restored_loss,best)

from sklearn.metrics import confusion_matrix,precision_recall_fscore_support
comparison=validation[['id','text','label']].copy()
comparison['prediccion']=prediction
comparison['acierto']=comparison.label.eq(comparison.prediccion)
display(comparison)
cm=confusion_matrix(validation.label,prediction,labels=LABELS)
display(pd.DataFrame(cm,index=['Real: '+s for s in LABELS],columns=['Pred: '+s for s in LABELS]))
precision,recall,f1,support=precision_recall_fscore_support(validation.label,prediction,labels=LABELS,zero_division=0)
display(pd.DataFrame({'clase':LABELS,'precision':precision,'recall':recall,'F1':f1,'casos_reales':support}))
print('Aciertos:',int(comparison.acierto.sum()),'de',len(comparison))

F1 macro validation: 0.2403846153846154


,id,text,label,prediccion,acierto
24,S025,La página regresa al inicio de sesión sin avisar.,acceso,software,False
25,S026,"Buen día, la página regresa al inicio de sesión sin avisar. ¿Me pueden orientar?",acceso,software,False
26,S027,"La página regresa al inicio de sesión sin avisar; ya reinicié y sigue igual, urge tantito.",acceso,software,False
27,S028,El autenticador muestra un código inválido.,acceso,software,False
28,S029,"Buen día, el autenticador muestra un código inválido. ¿Me pueden orientar?",acceso,software,False
29,S030,"El autenticador muestra un código inválido; ya reinicié y sigue igual, urge tantito.",acceso,software,False
60,S061,La suscripción figura como vencida aunque pagué.,pagos,software,False
61,S062,"Buen día, la suscripción figura como vencida aunque pagué. ¿Me pueden orientar?",pagos,software,False
62,S063,"La suscripción figura como vencida aunque pagué; ya reinicié y sigue igual, urge tantito.",pagos,software,False
63,S064,El banco confirmó el cargo pero no tengo recibo.,pagos,software,False


,Pred: acceso,Pred: conectividad,Pred: pagos,Pred: software
Real: acceso,0,0,0,6
Real: conectividad,2,2,0,2
Real: pagos,0,0,0,6
Real: software,0,0,0,6


,clase,precision,recall,F1,casos_reales
0,acceso,0.0,0.000000,0.000000,6
1,conectividad,1.0,0.333333,0.500000,6
2,pagos,0.0,0.000000,0.000000,6
3,software,0.3,1.000000,0.461538,6


Aciertos: 8 de 24


### Qué nos dice esta ejecución

La red acertó 8 de 24 mensajes: exactitud=8/24=0.333333≈33.33 %. En la matriz, filas son clases reales y columnas predichas; la diagonal contiene 0+2+0+6=8 aciertos. S097 («La red de invitados no permite navegar.») se clasificó como conectividad. S025 («La página regresa al inicio de sesión sin avisar.») era acceso y se predijo software. Estos son hechos observados; la causa del error requiere pruebas adicionales.

Para conectividad: TP=2, FP=0, FN=4. TP son aciertos de esa clase; FP, otros mensajes enviados a ella; FN, mensajes de ella enviados a otra. Precisión=2/(2+0)=1; recuperación o recall=2/(2+4)=1/3. F1=2×TP/(2×TP+FP+FN)=4/(4+0+4)=0.5. Para software: TP=6, FP=14, FN=0; precisión=6/20=0.3, recall=6/6=1 y F1=12/26≈0.461538. Acceso y pagos tienen F1=0; para pagos no hubo predicciones y declaramos precisión 0 mediante zero_division=0.

F1 macro=(0+0.5+0+0.461538)/4=0.240385. Da el mismo peso a cada clase. Es distinto de exactitud. Validation ya participó en elegir pesos: esta cifra no equivale a una evaluación final de test.

**Interpretación.** Validation participó en selección. Su F1 no es una evaluación final de test.

### Sobreajustar cuatro ejemplos

**Qué buscamos.** Entrenamos repetidamente un lote diminuto. Si no logra ajustarse, revisamos etiquetas, dimensiones, gradientes y tasa de aprendizaje.

**Qué no prueba.** Acertar cuatro ejemplos vistos no demuestra calidad en textos nuevos. La prueba revisa el funcionamiento del entrenamiento; después evaluamos por separado.

Generalizar significa funcionar con ejemplos no usados para ajustar pesos.

## 8. Sobreajustar cuatro ejemplos

Tomamos uno por clase y lo entrenamos repetidamente. Esto no mide generalización.

**Aquí recibimos:** Cuatro mensajes de train, uno por clase, y una red nueva.

**Lo que haremos:** Repetimos 60 actualizaciones en esos mismos ejemplos. Es una prueba de funcionamiento del entrenamiento.

**Antes de ejecutar:** ¿Acertar cuatro mensajes vistos demuestra que podrá atender cualquier solicitud?

In [9]:
tiny=train.groupby('label',sort=True).head(1);tx=encode(list(tiny.text),vocab);ty=torch.tensor([LABELS.index(z) for z in tiny.label])
small=MeanClassifier(len(vocab));opt=torch.optim.Adam(small.parameters(),lr=.08)
for step in range(60):
    opt.zero_grad();loss=torch.nn.functional.cross_entropy(small(tx),ty);loss.backward();opt.step()
print('Exactitud en lote de 4:',float((small(tx).argmax(-1)==ty).float().mean()))
display(tiny[["id","text","label"]])

Exactitud en lote de 4: 1.0


,id,text,label
0,S001,Olvidé la contraseña y necesito restablecerla.,acceso
36,S037,Me cobraron dos veces la misma factura.,pagos
72,S073,La red inalámbrica se desconecta cada cinco minutos.,conectividad
108,S109,La aplicación se cierra al abrir un archivo.,software


### Qué nos dice esta ejecución

Se ajustaron S001 (acceso), S037 (pagos), S073 (conectividad) y S109 (software). La red acertó los cuatro: 4/4=1, es decir, 100 % en el mismo lote utilizado para entrenarla. Esto comprueba que ese circuito pudo ajustar esos ejemplos. No contradice los 8/24 de validación: cambian modelo y conjunto observado.

**Interpretación.** Exactitud alta aquí solo acredita ajuste a este lote.

## E1. Reto del equipo

Compara con TF-IDF sobre la misma validación. Quita la máscara en una copia y explica cuánto cambia el promedio de un texto corto.

Trabajamos en equipos de cinco sobre un notebook común. Organicen las responsabilidades: coordinación y tiempo; revisión de datos; representación del texto; modelo y métricas; interpretación y límites. Cambien quién ejecuta y quién explica al pasar de bloque. Los cinco recorren toda la práctica y acuerdan las respuestas. Este reto es formativo: no añade calificación ni entrega oficial, ni cambia las evaluaciones individuales del curso.

**Antes de ejecutar:** escriban qué esperan que cambie. **Después:** citen una salida concreta, expliquen una limitación y acuerden qué decisión tomarían.

**Nuestra predicción:** (1) TF-IDF + regresión logística superará a la red en F1 macro de validation (la red tiene 0.24): con 96 textos de entrenamiento, un modelo lineal sobre palabras y bigramas suele ser una referencia fuerte, y la red de promedio de embeddings aprendidos desde cero tiene poco dato para formar buenos vectores. Aun así, ambos sufrirán porque validation usa familias de mensajes que no se vieron en train. (2) Sin máscara, el vector promedio de un texto corto **se encoge** por un factor igual a (palabras reales)/32; como el embedding de padding vale cero, la suma es la misma y solo cambia el denominador. Para un texto de ~10 palabras, el promedio queda en ~31 % de su valor correcto.

**Evidencia y conclusión del equipo:** la celda de código de abajo calcula (a) F1 macro y exactitud de TF-IDF+logística frente a la red, con matriz de confusión de la referencia; y (b) para el texto más corto de validation: número de palabras reales, razón de normas sin/con máscara, comprobación de que `sin_máscara = con_máscara × n/32`, y cuántas predicciones de validation cambian si se aplica la media sin máscara al modelo ya entrenado. **Limitaciones:** validation ya se usó para elegir la época de la red, y la referencia no la usó para nada, así que la comparación es justa solo a medias; con 24 mensajes (6 por clase, en 8 familias de 3 variantes) una diferencia de 1–2 aciertos no es concluyente; y quitar la máscara *solo en inferencia* (sin reentrenar) mide cuánto se distorsiona la representación, no cuánto empeoraría un modelo entrenado sin máscara. **Decisión:** conservar la máscara (es correcta y barata) y tomar a TF-IDF+logística como referencia mínima que cualquier red debe superar antes de justificar su complejidad; evaluar en test solo al final.


**Aquí recibimos:** train y validation sin cambios, la referencia clásica y una palabra codificada con relleno.

**Lo que haremos:** Comparamos F1 en la misma validación y estudiamos qué cambia al quitar la máscara.

**Antes de ejecutar:** ¿Qué explicación de los resultados sería más útil que decir ganó la red?

In [10]:
# E1 (parte 1): referencia clásica sobre la MISMA validación. Se ajusta solo con train.
from sklearn.metrics import accuracy_score
from apoyo_pln import make_baseline, evaluate

base = make_baseline("logistic").fit(train.text, train.label)
res_base = evaluate(base, validation)
pred_base = res_base["predictions"]

display(pd.DataFrame({
    "modelo": ["TF-IDF + regresión logística", "Red de media con máscara (época 7)"],
    "F1 macro": [res_base["macro_f1"], f1_score(validation.label, prediction, average="macro")],
    "exactitud": [accuracy_score(validation.label, pred_base), accuracy_score(validation.label, prediction)],
}).round(3))
display(pd.DataFrame(res_base["confusion"],
                     index=["Real: " + s for s in LABELS], columns=["Pred: " + s for s in LABELS]))

# E1 (parte 2): quitar la máscara en una copia (solo inferencia, sin reentrenar).
def pooled_con_mascara(m, ids):
    valid = (ids != 0).unsqueeze(-1)
    return (m.embedding(ids) * valid).sum(1) / valid.sum(1).clamp(min=1)

def pooled_sin_mascara(m, ids):
    return m.embedding(ids).mean(1)          # divide entre las 32 posiciones, reales o no

with torch.no_grad():
    print("¿La fila de padding del embedding vale cero?", bool((model.embedding.weight[0] == 0).all()))
    n_reales = (vx != 0).sum(1)
    i = int(n_reales.argmin())                # texto más corto de validation
    con, sin = pooled_con_mascara(model, vx[i:i+1]), pooled_sin_mascara(model, vx[i:i+1])
    n = int(n_reales[i])
    print(f"Texto {validation.id.iloc[i]}: «{validation.text.iloc[i]}»  ->  {n} tokens reales de {vx.shape[1]}")
    print("Razón de normas sin/con máscara:", round(float(sin.norm() / con.norm()), 4), "| n/32 =", round(n / vx.shape[1], 4))
    assert torch.allclose(sin, con * n / vx.shape[1], atol=1e-6)

    # Efecto en todas las predicciones de validation con el modelo ya entrenado
    pred_sin = model.output(pooled_sin_mascara(model, vx)).argmax(-1)
    pred_con = model.output(pooled_con_mascara(model, vx)).argmax(-1)
    print("Predicciones que cambian al quitar la máscara:", int((pred_sin != pred_con).sum()), "de", len(vx))
    print("F1 macro sin máscara:", round(f1_score(validation.label, [LABELS[int(j)] for j in pred_sin], average="macro"), 3),
          "| con máscara:", round(f1_score(validation.label, prediction, average="macro"), 3))


,modelo,F1 macro,exactitud
0,TF-IDF + regresión logística,0.432,0.500
1,Red de media con máscara (época 7),0.240,0.333


,Pred: acceso,Pred: conectividad,Pred: pagos,Pred: software
Real: acceso,3,0,0,3
Real: conectividad,0,6,0,0
Real: pagos,0,5,0,1
Real: software,0,3,0,3


¿La fila de padding del embedding vale cero? True
Texto S028: «El autenticador muestra un código inválido.»  ->  7 tokens reales de 32
Razón de normas sin/con máscara: 0.2187 | n/32 = 0.2188
Predicciones que cambian al quitar la máscara: 5 de 24
F1 macro sin máscara: 0.08 | con máscara: 0.24


**Mi explicación:**

- **Qué conservé:** train/validation sin cambios, la red entrenada (época 7 restaurada), las mismas 24 predicciones de validation y el orden de `LABELS`.
- **Qué cambié:** (1) añadí un modelo clásico (TF-IDF con unigramas y bigramas + regresión logística, ajustado solo con train); (2) en una copia de la inferencia reemplacé la media con máscara por `mean(1)`, que divide entre las 32 posiciones.
- **Qué observé:** ver la celda de arriba. Matemáticamente, como el vector de padding es cero, `media_sin_máscara = media_con_máscara × n/32`: el texto corto conserva su *dirección* pero pierde *magnitud* (con n≈10 queda en ~31 %). Como el clasificador final es lineal con sesgo, encoger el vector acerca los logits al sesgo y puede cambiar la clase predicha; por eso se cuentan cuántas predicciones cambian. Un texto largo (cerca de 32 tokens) casi no se vería afectado: el error depende de la longitud, lo que además mezcla longitud del mensaje con su contenido.
- **Qué todavía no puedo concluir:** que «TF-IDF gana» o «la red gana» de forma general (24 mensajes de validation y una sola ejecución); que quitar la máscara empeore un modelo *entrenado* sin ella (aquí solo la quité al predecir); ni nada sobre test, que sigue sin tocarse.
- **Decisión que tomaría:** usar TF-IDF + logística como línea base obligatoria, mantener la máscara, y solo evaluar en test una vez elegido el modelo final.


## Cierre

Conserva salidas de tu ejecución. Los ejemplos numéricos enseñan el mecanismo; los experimentos conservan corpus, IDs, etiquetas y particiones. Una diferencia exige revisar configuración antes de proponer una causa.

Referencia: https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html

In [11]:
# Dejamos constancia del entorno que produjo las salidas anteriores.
import sys, importlib.metadata as _metadata
from datetime import datetime, timezone
print('EJECUCION_COMPLETA_CLASE_08_ALUMNO')
print('Fecha UTC:', datetime.now(timezone.utc).isoformat())
print('Python:', sys.version.split()[0])
for _lib in ['numpy', 'pandas', 'scikit-learn', 'torch']:
    try: print(_lib + ':', _metadata.version(_lib))
    except _metadata.PackageNotFoundError: pass

EJECUCION_COMPLETA_CLASE_08_ALUMNO
Fecha UTC: 2026-10-31T18:06:46.220930+00:00
Python: 3.12.10
numpy: 2.5.3
pandas: 3.0.5
scikit-learn: 1.9.0
torch: 2.15.0
